In [ ]:

"""
Project: ATM and UPI Transaction Simulator
Purpose: Simulate basic ATM and UPI banking operations.
"""

import os
from getpass import getpass


# --------------------------------------------------
# 1. INITIAL SETTINGS
# --------------------------------------------------

PIN_FILE = "pin.txt"

# Load the saved PIN or use the default PIN.
if os.path.exists(PIN_FILE):
    with open(PIN_FILE, "r") as file:
        correct_pin = file.read().strip()

    # Check whether the saved PIN is valid.
    if len(correct_pin) != 4 or not correct_pin.isdigit():
        correct_pin = "1234"
else:
    correct_pin = "1234"

balance = 10000.00

minimum_balance = 500.00
daily_withdrawal_limit = 20000.00
upi_limit = 5000.00
large_transfer_limit = 3000.00

daily_withdrawn = 0.00
large_transfer_streak = 0

denominations = (500, 200, 100)

# Each transaction is stored as a tuple inside this list.
transactions = []


# --------------------------------------------------
# 2. TRANSACTION RECORDING FUNCTION
# --------------------------------------------------

def record_transaction(transaction_type, amount):
    """Store transaction details in a tuple."""

    transaction_id = len(transactions) + 1

    transaction = (
        transaction_id,
        transaction_type,
        amount,
        balance
    )

    transactions.append(transaction)


# --------------------------------------------------
# 3. PIN VERIFICATION
# --------------------------------------------------

login_successful = False

print("\n===================================")
print("       WELCOME TO ABC BANK")
print("===================================")

for attempt in range(1, 4):

    entered_pin = getpass("Enter your 4-digit PIN: ")

    if entered_pin == correct_pin:
        print("\nLogin successful!")
        login_successful = True
        break

    else:
        attempts_left = 3 - attempt
        print("Incorrect PIN.")

        if attempts_left > 0:
            print("Attempts remaining:", attempts_left)

else:
    print("\nYour card has been blocked.")
    print("You have entered the wrong PIN 3 times.")

# The for-else block runs only when the loop finishes
# without executing break.


# --------------------------------------------------
# 4. ATM MENU
# --------------------------------------------------

if login_successful:

    while True:

        print("\n===================================")
        print("             ATM MENU")
        print("===================================")

        print("1. Balance Enquiry")
        print("2. Cash Withdrawal")
        print("3. Deposit")
        print("4. UPI Transfer")
        print("5. Mini-Statement")
        print("6. Change PIN")
        print("7. Exit")

        print("===================================")

        choice = input("Enter your choice: ").strip()

        # Check for invalid menu choices.
        if choice not in ("1", "2", "3", "4", "5", "6", "7"):
            print("Invalid menu choice. Try again.")
            large_transfer_streak = 0
            continue

        # Any non-UPI menu option breaks the
        # consecutive large UPI transfer sequence.
        if choice != "4":
            large_transfer_streak = 0


        # ------------------------------------------
        # OPTION 1: BALANCE ENQUIRY
        # ------------------------------------------

        if choice == "1":

            print("\nCurrent balance: Rs.", balance)


        # ------------------------------------------
        # OPTION 2: CASH WITHDRAWAL
        # ------------------------------------------

        elif choice == "2":

            try:
                amount = float(input("Enter withdrawal amount: Rs. "))

            except ValueError:
                print("Invalid input. Enter a numeric amount.")
                continue

            if amount <= 0:
                print("Amount must be greater than zero.")
                continue

            if amount != int(amount) or int(amount) % 100 != 0:
                print("Invalid amount. Enter multiples of Rs. 100.")
                continue

            amount = int(amount)

            # Check daily withdrawal limit.
            if daily_withdrawn + amount > daily_withdrawal_limit:
                print("Daily withdrawal limit exceeded.")
                continue

            # Check minimum balance requirement.
            if balance - amount < minimum_balance:
                print("Insufficient balance.")
                print("You must maintain Rs. 500 minimum balance.")
                continue

            # Cash dispenser algorithm.
            remaining = amount
            cash_notes = []

            for note in denominations:

                count = 0

                while remaining >= note:
                    remaining = remaining - note
                    count = count + 1

                if count > 0:
                    cash_notes.append((note, count))

                if remaining == 0:
                    break

            # Check whether the amount can be dispensed.
            if remaining != 0:
                print("Sorry, this amount cannot be dispensed.")
                continue

            # Update balance and daily withdrawal.
            balance = balance - amount
            daily_withdrawn = daily_withdrawn + amount

            record_transaction("Cash Withdrawal", amount)

            print("\nWithdrawal successful!")
            print("Amount withdrawn: Rs.", amount)

            print("\nCash denominations:")

            for note, count in cash_notes:
                print("Rs.", note, "x", count)

            print("Remaining balance: Rs.", balance)


        # ------------------------------------------
        # OPTION 3: DEPOSIT
        # ------------------------------------------

        elif choice == "3":

            try:
                amount = float(input("Enter deposit amount: Rs. "))

            except ValueError:
                print("Invalid input. Enter a numeric amount.")
                continue

            if amount <= 0:
                print("Deposit amount must be greater than zero.")
                continue

            balance = balance + amount

            record_transaction("Deposit", amount)

            print("\nDeposit successful!")
            print("Amount deposited: Rs.", amount)
            print("Updated balance: Rs.", balance)


        # ------------------------------------------
        # OPTION 4: UPI TRANSFER
        # ------------------------------------------

        elif choice == "4":

            upi_id = input("Enter receiver's UPI ID: ").strip()

            # Basic UPI ID format validation.
            if (
                "@" not in upi_id
                or upi_id.startswith("@")
                or upi_id.endswith("@")
                or " " in upi_id
            ):
                print("Invalid UPI ID.")
                large_transfer_streak = 0
                continue

            try:
                amount = float(input("Enter transfer amount: Rs. "))

            except ValueError:
                print("Invalid input. Enter a numeric amount.")
                large_transfer_streak = 0
                continue

            if amount <= 0:
                print("Transfer amount must be greater than zero.")
                large_transfer_streak = 0
                continue

            if amount > upi_limit:
                print("UPI limit exceeded.")
                print("Maximum transfer per transaction is Rs. 5000.")
                large_transfer_streak = 0
                continue

            if balance - amount < minimum_balance:
                print("Insufficient balance.")
                print("Minimum balance of Rs. 500 must be maintained.")
                large_transfer_streak = 0
                continue

            # Complete the UPI transfer.
            balance = balance - amount

            record_transaction("UPI Transfer", amount)

            print("\nUPI transfer successful!")
            print("Receiver:", upi_id)
            print("Amount transferred: Rs.", amount)
            print("Remaining balance: Rs.", balance)

            # Check for consecutive large transfers.
            if amount >= large_transfer_limit:

                large_transfer_streak = large_transfer_streak + 1

                if large_transfer_streak == 3:
                    print("\nFRAUD ALERT!")
                    print("Three consecutive large UPI transfers detected.")
                    large_transfer_streak = 0

            else:
                large_transfer_streak = 0


        # ------------------------------------------
        # OPTION 5: MINI-STATEMENT
        # ------------------------------------------

        elif choice == "5":

            print("\n========== MINI-STATEMENT ==========")

            if len(transactions) == 0:
                print("No transactions available.")

            else:
                # Display the last five transactions.
                recent_transactions = transactions[-5:]

                print("ID\tType\t\tAmount\t\tBalance")
                print("---------------------------------------------")

                for transaction in recent_transactions:

                    transaction_id, transaction_type, amount, current_balance = transaction

                    print(
                        transaction_id,
                        "\t",
                        transaction_type,
                        "\t",
                        "Rs.", amount,
                        "\t",
                        "Rs.", current_balance
                    )

            print("---------------------------------------------")


        # ------------------------------------------
        # OPTION 6: CHANGE PIN
        # ------------------------------------------

        elif choice == "6":

            current_pin = getpass("Enter current PIN: ")

            if current_pin != correct_pin:
                print("Incorrect current PIN.")
                continue

            new_pin = getpass("Enter new 4-digit PIN: ")

            if len(new_pin) != 4 or not new_pin.isdigit():
                print("PIN must contain exactly 4 digits.")
                continue

            if new_pin == correct_pin:
                print("New PIN must be different from the old PIN.")
                continue

            confirm_pin = getpass("Confirm new PIN: ")

            if new_pin != confirm_pin:
                print("PIN confirmation does not match.")
                continue

            # Update and save the new PIN.
            correct_pin = new_pin

            with open(PIN_FILE, "w") as file:
                file.write(correct_pin)

            record_transaction("PIN Change", 0)

            print("\nPIN changed successfully!")
            print("Your new PIN has been saved.")


        # ------------------------------------------
        # OPTION 7: EXIT
        # ------------------------------------------

        elif choice == "7":

            print("\nThank you for using ABC Bank!")
            print("Please visit again.")
            break



       WELCOME TO ABC BANK
